# IEEE-CIS Fraud Detection — Kaggle Automation

Runs the 7 model_experiment notebooks (and EDA + inference) sequentially via nbclient,
then auto-registers the best model and generates the Kaggle submission.

The 7 notebooks themselves are not modified — each logs to its own MLflow experiment on
DagsHub, exactly as if you ran them individually.

**Setup before running:**
1. Attach the `ieee-fraud-detection` competition dataset (Add-ons -> Datasets).
2. Add a Kaggle secret `DAGSHUB_TOKEN` with your DagsHub access token, toggle Attached on.

In [ ]:
import os, sys, subprocess, warnings, logging
warnings.filterwarnings('ignore')
logging.getLogger('mlflow').setLevel(logging.ERROR)

subprocess.run(['pip', 'install', '-q', 'dagshub', 'mlflow', 'xgboost', 'nbclient'], check=True)

REPO_DIR = '/kaggle/working/ML_Asgn2'
if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--quiet'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/Saba0033/ML_Asgn2.git', REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

from kaggle_secrets import UserSecretsClient
os.environ['DAGSHUB_USER_TOKEN'] = UserSecretsClient().get_secret('DAGSHUB_TOKEN')

print('Bootstrap complete.')

In [ ]:
import nbformat
from nbclient import NotebookClient
from nbclient.exceptions import CellExecutionError

NOTEBOOKS = [
    '00_eda.ipynb',
    'model_experiment_LogisticRegression.ipynb',
    'model_experiment_LogisticRegression_L1.ipynb',
    'model_experiment_DecisionTree.ipynb',
    'model_experiment_RandomForest.ipynb',
    'model_experiment_AdaBoost.ipynb',
    'model_experiment_GradientBoosting.ipynb',
    'model_experiment_XGBoost.ipynb',
]

for path in NOTEBOOKS:
    print(f'\n{"="*70}\nRunning {path}\n{"="*70}')
    nb = nbformat.read(path, as_version=4)
    client = NotebookClient(nb, timeout=3600, kernel_name='python3',
                            resources={'metadata': {'path': REPO_DIR}})
    try:
        client.execute()
    except CellExecutionError as e:
        print(f'  FAILED: {e}')
        raise
    nbformat.write(nb, path)
    print(f'  done.')

print('\nAll notebooks complete.')

In [ ]:
import json, mlflow
import dagshub

dagshub.init(repo_owner='Saba0033', repo_name='ML_Asgn2', mlflow=True)

with open('results_cache.json') as f:
    data = json.load(f)

print('\nResults summary:')
print('-' * 70)
sorted_arch = sorted(data, key=lambda a: data[a]['cv_val_roc_auc_mean'], reverse=True)
for arch in sorted_arch:
    d = data[arch]
    print(f"  {arch:25s}  CV={d['cv_val_roc_auc_mean']:.4f}  gap={d['overfit_gap']:+.4f}  "
          f"sel={d['best_selector']}  feats={d['n_features_kept']}")

winner_name = sorted_arch[0]
winner = data[winner_name]
print(f'\nWinner: {winner_name}  (CV ROC-AUC = {winner["cv_val_roc_auc_mean"]:.4f})')
print(f'  best_params: {winner["best_params"]}')
print(f'  final_run_id: {winner["final_run_id"]}')

mv = mlflow.register_model(
    model_uri=f"runs:/{winner['final_run_id']}/pipeline",
    name='IEEEFraudBestModel',
)
print(f'\nRegistered IEEEFraudBestModel version {mv.version}')

In [ ]:
import nbformat
from nbclient import NotebookClient

print('Running model_inference.ipynb...')
nb = nbformat.read('model_inference.ipynb', as_version=4)
NotebookClient(nb, timeout=1800, kernel_name='python3',
               resources={'metadata': {'path': REPO_DIR}}).execute()
nbformat.write(nb, 'model_inference.ipynb')

import pandas as pd
sub = pd.read_csv('submissions/submission.csv')
print(f'\nsubmission.csv: {len(sub):,} rows')
print(sub.head())
print('\nDownload from /kaggle/working/ML_Asgn2/submissions/submission.csv and submit:')
print('  kaggle competitions submit -c ieee-fraud-detection -f submission.csv -m "best model"')